In [1]:
# pycodestyle_magic for pep8 compliance
%load_ext pycodestyle_magic
# Activate pycodestyle
%pycodestyle_on

# Projet 7 : Implémentez un modèle de scoring

# Construction des jeux de données & feature engineering

## Mise en place

### Imports de module

In [2]:
import gc
import time
import pickle
import numpy as np
import pandas as pd
from contextlib import contextmanager
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

### Définitions de fonctions

In [3]:
@contextmanager
def timer(title):
    t0 = time.time()
    print(f'{title} - starting')
    yield
    print('{} - done in {:.0f}s'.format(title, time.time() - t0))


def is_int(s):
    return sum(s.dropna() % 1) == 0.0


def describe_numeric(df, sortByMissingFreq=False):
    desDf = df.describe(include='number').transpose()
    desDf['count'] = desDf['count'].astype('int64')
    tmpS = (1 - desDf['count'] / df.shape[0]) * 100
    tmpS = tmpS.astype('float64').round(3)
    desDf.insert(desDf.columns.to_list().index('count') + 1,
                 'missing freq',
                 tmpS,
                 allow_duplicates=False)
    tmpS = desDf.apply(lambda x: 'int' if is_int(df[x.name]) else 'float',
                       axis='columns')
    desDf.insert(desDf.columns.to_list().index('missing freq') + 1,
                 'inferred type',
                 tmpS,
                 allow_duplicates=False)
    desDf.rename(columns={
        '50%': '50% / median',
    }, inplace=True)
    if sortByMissingFreq:
        desDf.sort_values(by='count',
                          inplace=True)
    return desDf


def describe_int(df, sortByMissingFreq=False):
    desDf = df.describe(include='number').transpose()
    desDf = desDf.loc[[col for col in desDf.index if is_int(df[col])]]
    desDf['count'] = desDf['count'].astype('int64')
    desDf['min'] = desDf['min'].astype('int64')
    desDf['25%'] = desDf['25%'].astype('int64')
    desDf['50%'] = desDf['50%'].astype('int64')
    desDf['75%'] = desDf['75%'].astype('int64')
    desDf['max'] = desDf['max'].astype('int64')
    tmpS = (1 - desDf['count'] / df.shape[0]) * 100
    tmpS = tmpS.astype('float64').round(3)
    desDf.insert(desDf.columns.to_list().index('count') + 1,
                 'missing freq',
                 tmpS,
                 allow_duplicates=False)
    desDf.insert(desDf.columns.to_list().index('missing freq') + 1,
                 'unique',
                 df[desDf.index].nunique(axis='index'),
                 allow_duplicates=False)
    tmpS = df[desDf.index].mode(dropna=True).iloc[0]
    tmpS = tmpS.transpose().astype('int64')
    desDf.insert(desDf.columns.to_list().index('unique') + 1,
                 'mode',
                 tmpS,
                 allow_duplicates=False)
    tmpS = df[desDf.index].apply(lambda x: (x == tmpS[x.name]).sum(),
                                 axis='index')
    desDf.insert(desDf.columns.to_list().index('mode') + 1,
                 'mode count',
                 tmpS,
                 allow_duplicates=False)
    tmpS = desDf['mode count'] / desDf['count'] * 100
    tmpS = tmpS.astype('float64').round(3)
    desDf.insert(desDf.columns.to_list().index('mode count') + 1,
                 'mode freq',
                 tmpS,
                 allow_duplicates=False)
    if sortByMissingFreq:
        desDf.sort_values(by='count',
                          inplace=True)
    return desDf


def describe_float(df, sortByMissingFreq=False):
    desDf = df.describe(include='number').transpose()
    desDf = desDf.loc[[col for col in desDf.index if not is_int(df[col])]]
    desDf['count'] = desDf['count'].astype('int64')
    tmpS = (1 - desDf['count'] / df.shape[0]) * 100
    tmpS = tmpS.astype('float64').round(3)
    desDf.insert(desDf.columns.to_list().index('count') + 1,
                 'missing freq',
                 tmpS,
                 allow_duplicates=False)
    if sortByMissingFreq:
        desDf.sort_values(by='count',
                          inplace=True)
    return desDf


def describe_categorical(df, sortByMissingFreq=False):
    desDf = df.describe(include='object').transpose()
    tmpS = (1 - desDf['count'] / df.shape[0]) * 100
    tmpS = tmpS.astype('float64').round(3)
    desDf.insert(desDf.columns.to_list().index('count') + 1,
                 'missing freq',
                 tmpS,
                 allow_duplicates=False)
    tmpS = desDf['freq'] / desDf['count'] * 100
    tmpS = tmpS.astype('float64').round(3)
    desDf['mode freq'] = tmpS
    desDf.rename(columns={
        'top': 'mode',
        'freq': 'mode count',
    }, inplace=True)
    tmpS = desDf.apply(lambda x: df[x.name].dropna().unique()[:10],
                       axis='columns')
    desDf['value samples'] = tmpS
    if sortByMissingFreq:
        desDf.sort_values(by='count',
                          inplace=True)
    return desDf


def one_hot_name_combiner(input_feature, category):
    return 'ONEHOT__' + input_feature + '__' + str(category)

### Initialisation

In [4]:
# Initialisation de l'affichage pandas
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', None)

# Cache to save encoders
context = {}

## Création du jeu de données de base

### Fichiers application (application_train & application_test) 

In [5]:
apDf = pd.read_csv(r'data\application_train.csv', low_memory=False)
testApDf = pd.read_csv(r'data\application_test.csv', low_memory=False)
print('Train samples: {}, test samples: {}'.format(len(apDf),
                                                   len(testApDf)))
apDf = pd.concat([apDf, testApDf], axis='index').reset_index(drop=True)

Train samples: 307511, test samples: 48744


In [6]:
apDf.set_index('SK_ID_CURR', drop=True, inplace=True)
print('Index set to SK_ID_CURR, check unicity:', apDf.index.is_unique)

Index set to SK_ID_CURR, check unicity: True


In [7]:
describe_categorical(apDf, True)

,count,missing freq,unique,mode,mode count,mode freq,value samples
FONDKAPREMONT_MODE,113163,68.235,4,reg oper account,85954,75.956,"[reg oper account, org spec account, reg oper spec account, not specified]"
WALLSMATERIAL_MODE,176021,50.591,7,Panel,77309,43.920,"[Stone, brick, Block, Panel, Mixed, Wooden, Others, Monolithic]"
HOUSETYPE_MODE,178339,49.941,3,block of flats,175162,98.219,"[block of flats, terraced house, specific housing]"
EMERGENCYSTATE_MODE,188291,47.147,2,No,185607,98.575,"[No, Yes]"
OCCUPATION_TYPE,244259,31.437,18,Laborers,63841,26.137,"[Laborers, Core staff, Accountants, Managers, Drivers, Sales staff, Cleaning staff, Cooking staff, Private service staff, Medicine staff]"
NAME_TYPE_SUITE,354052,0.618,7,Unaccompanied,288253,81.415,"[Unaccompanied, Family, Spouse, partner, Children, Other_A, Other_B, Group of people]"
NAME_CONTRACT_TYPE,356255,0.000,2,Cash loans,326537,91.658,"[Cash loans, Revolving loans]"
CODE_GENDER,356255,0.000,3,F,235126,65.999,"[M, F, XNA]"
FLAG_OWN_CAR,356255,0.000,2,N,235235,66.030,"[N, Y]"
FLAG_OWN_REALTY,356255,0.000,2,Y,246970,69.324,"[Y, N]"


In [8]:
describe_int(apDf, True)

,count,missing freq,unique,mode,mode count,mode freq,mean,std,min,25%,50%,75%,max
OWN_CAR_AGE,121014,66.032,63,7,8625,7.127,12.023741,11.880848,0,5,9,15,91
TARGET,307511,13.682,2,0,282686,91.927,0.080729,0.272419,0,0,0,0,1
AMT_REQ_CREDIT_BUREAU_DAY,308687,13.352,9,0,307128,99.495,0.006281,0.104250,0,0,0,0,9
AMT_REQ_CREDIT_BUREAU_QRT,308687,13.352,11,0,238976,77.417,0.304399,0.786915,0,0,0,0,261
AMT_REQ_CREDIT_BUREAU_YEAR,308687,13.352,25,0,82640,26.771,1.911564,1.865338,0,0,1,3,25
AMT_REQ_CREDIT_BUREAU_WEEK,308687,13.352,9,0,300035,97.197,0.029995,0.191374,0,0,0,0,8
AMT_REQ_CREDIT_BUREAU_MON,308687,13.352,24,0,264574,85.709,0.231697,0.855949,0,0,0,0,27
AMT_REQ_CREDIT_BUREAU_HOUR,308687,13.352,5,0,306972,99.444,0.005808,0.079736,0,0,0,0,4
OBS_60_CNT_SOCIAL_CIRCLE,355205,0.295,35,0,190806,53.717,1.409468,2.577724,0,0,0,2,351
DEF_60_CNT_SOCIAL_CIRCLE,355205,0.295,9,0,325335,91.591,0.100198,0.368259,0,0,0,0,24


In [9]:
describe_float(apDf, True)

,count,missing freq,mean,std,min,25%,50%,75%,max
COMMONAREA_AVG,107895,69.714,0.045045,0.077045,0.000000e+00,0.007900,0.021300,0.051900,1.000000e+00
COMMONAREA_MODE,107895,69.714,0.042930,0.075437,0.000000e+00,0.007300,0.019200,0.049300,1.000000e+00
COMMONAREA_MEDI,107895,69.714,0.044994,0.077140,0.000000e+00,0.007900,0.021000,0.051800,1.000000e+00
NONLIVINGAPARTMENTS_AVG,109394,69.293,0.008868,0.047876,0.000000e+00,0.000000,0.000000,0.003900,1.000000e+00
NONLIVINGAPARTMENTS_MODE,109394,69.293,0.008116,0.046330,0.000000e+00,0.000000,0.000000,0.003900,1.000000e+00
NONLIVINGAPARTMENTS_MEDI,109394,69.293,0.008697,0.047519,0.000000e+00,0.000000,0.000000,0.003900,1.000000e+00
LIVINGAPARTMENTS_MEDI,113276,68.204,0.102674,0.094541,0.000000e+00,0.051300,0.077000,0.123100,1.000000e+00
LIVINGAPARTMENTS_AVG,113276,68.204,0.101495,0.093418,0.000000e+00,0.050400,0.075600,0.121000,1.000000e+00
LIVINGAPARTMENTS_MODE,113276,68.204,0.106382,0.098779,0.000000e+00,0.054200,0.077100,0.131300,1.000000e+00
FLOORSMIN_AVG,115147,67.678,0.232817,0.161909,0.000000e+00,0.083300,0.208300,0.375000,1.000000e+00


#### Nettoyage & encodage catégoriel

In [10]:
def application_cleaning_encoding(df,
                                  context=None,
                                  verbose=True):

    # Local overriding of print
    def _print(*args, **kwargs):
        if 'tabs' in kwargs:
            tabs = int(kwargs['tabs'])
            del kwargs['tabs']
        else:
            tabs = 2
        if verbose:
            print('\t'*tabs, *args, **kwargs)
        else:
            pass

    _print('Cleaning & encoding of application train and test', tabs=1)

    labeled = ~df['TARGET'].isna()

    # Handling CODE_GENDER = XNA
    trnN = df.loc[labeled & (df['CODE_GENDER'] == 'XNA'), :].shape[0]
    tstN = df.loc[~labeled & (df['CODE_GENDER'] == 'XNA'), :].shape[0]
    _print('Number of applications with CODE_GENDER=XNA (train/test):',
           str(trnN) + '/' + str(tstN))
    _print(f'Removing {trnN + tstN} application(s) with CODE_GENDER=XNA')
    df = df.loc[df['CODE_GENDER'] != 'XNA', :]

    # NaN values for DAYS_EMPLOYED which value is 365243 (outlier)
    trnN = df.loc[labeled & (df['DAYS_EMPLOYED'] == 365243), :].shape[0]
    tstN = df.loc[~labeled & (df['DAYS_EMPLOYED'] == 365243), :].shape[0]
    _print('Number of applications with DAYS_EMPLOYED=365243 (train/test):',
           str(trnN) + '/' + str(tstN))
    _print(f'Updating {trnN + tstN} application(s),',
           'setting DAYS_EMPLOYED from 365243 to NaN')
    df.loc[df['DAYS_EMPLOYED'] == 365243, 'DAYS_EMPLOYED'] = pd.NA

    # Potentially create & fit the encoders for the categorical features
    if context is None:
        context = {}
        desc = None
    # Encoder for 2-class categorical features
    if 'app_bin_cat_features_encoder' not in context:
        # Get a description of the categorical features
        desc = describe_categorical(df)
        # List of binary categorical features
        binCatCols = desc[desc['unique'] == 2].index.to_list()
        # Create & fit an ordinal encoder => encoding to 0, 1 or NaN
        encoder = OrdinalEncoder(handle_unknown='error',
                                 dtype=np.float64).fit(df[binCatCols])
        context['app_bin_cat_features_encoder'] = encoder
    # Encoder for multi-class categorical features
    if 'app_multi_cat_features_encoder' not in context:
        if desc is None:
            # Get a description of the categorical features
            desc = describe_categorical(df)
        # List of multi-class categorical features
        multiCatCols = desc[desc['unique'] > 2].index.to_list()
        # Create & fit a one hot encoder => encoding to 0, 1 onto several cols
        encoder = OneHotEncoder(drop=None,
                                handle_unknown='error',
                                sparse_output=False,
                                dtype=np.float64,
                                feature_name_combiner=one_hot_name_combiner)
        encoder.fit(df[multiCatCols])
        context['app_multi_cat_features_encoder'] = encoder

    # Encoding binary categorical features (0, 1 or NaN)
    encoder = context['app_bin_cat_features_encoder']
    tmpData = encoder.transform(df[encoder.feature_names_in_])
    df.loc[:, 'BINARY__' + encoder.feature_names_in_] = tmpData

    # Encoding multi-class categorical features (n features with value 0 or 1)
    # Note that missing values are one-hot encoded in specific features
    encoder = context['app_multi_cat_features_encoder']
    tmpData = encoder.transform(df[encoder.feature_names_in_])
    df.loc[:, encoder.get_feature_names_out()] = tmpData

    # Defragmenting the DataFrame
    df = df.copy()

    return df, context

#### Imputation

In [11]:
def application_imputation(df,
                           context=None,
                           verbose=True):

    # Local overriding of print
    def _print(*args, **kwargs):
        if 'tabs' in kwargs:
            tabs = int(kwargs['tabs'])
            del kwargs['tabs']
        else:
            tabs = 2
        if verbose:
            print('\t'*tabs, *args, **kwargs)
        else:
            pass

    _print('No imputation for application train and test', tabs=1)

    return df, context

#### Feature engineering

In [12]:
def application_feature_engineering(df,
                                    context=None,
                                    verbose=True):

    # Local overriding of print
    def _print(*args, **kwargs):
        if 'tabs' in kwargs:
            tabs = int(kwargs['tabs'])
            del kwargs['tabs']
        else:
            tabs = 2
        if verbose:
            print('\t'*tabs, *args, **kwargs)
        else:
            pass

    _print('Feature engineering for application train and test', tabs=1)

    # Some simple new features
    _print('Creating feature: Number of Employment days in %')
    df['DAYS_EMPLOYED_PERC'] = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']
    _print('Creating feature: Income / credit ratio in %')
    df['INCOME_CREDIT_PERC'] = df['AMT_INCOME_TOTAL'] / df['AMT_CREDIT']
    _print('Creating feature: Income per family members in income unit')
    df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']
    _print('Creating feature: Loan annuity / income ratio in %')
    df['ANNUITY_INCOME_PERC'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
    _print('Creating feature: Loan annuity / credit ratio in %')
    df['PAYMENT_RATE'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']

    # Defragmenting the DataFrame
    df = df.copy()

    return df, context

#### Traitement

In [13]:
with timer('Process application_train & application_test'):
    apDf, context = application_cleaning_encoding(apDf,
                                                  context,
                                                  True)
    apDf, context = application_imputation(apDf,
                                           context,
                                           True)
    apDf, context = application_feature_engineering(apDf,
                                                    context,
                                                    True)
    # Freeing memory
    gc.collect()
print('DataFrame shape:', apDf.shape)

Process application_train & application_test - starting
	 Cleaning & encoding of application train and test
		 Number of applications with CODE_GENDER=XNA (train/test): 4/0
		 Removing 4 application(s) with CODE_GENDER=XNA
		 Number of applications with DAYS_EMPLOYED=365243 (train/test): 55374/9274
		 Updating 64648 application(s), setting DAYS_EMPLOYED from 365243 to NaN
	 No imputation for application train and test
	 Feature engineering for application train and test
		 Creating feature: Number of Employment days in %
		 Creating feature: Income / credit ratio in %
		 Creating feature: Income per family members in income unit
		 Creating feature: Loan annuity / income ratio in %
		 Creating feature: Loan annuity / credit ratio in %
Process application_train & application_test - done in 7s
DataFrame shape: (356251, 265)


### Fichiers bureau (bureau & bureau_balance)

In [14]:
brDf = pd.read_csv(r'data\bureau.csv', low_memory=False)
bbDf = pd.read_csv(r'data\bureau_balance.csv', low_memory=False)
print('Bureau samples: {}'.format(len(brDf)))
print('Bureau balance samples: {}'.format(len(bbDf)))

Bureau samples: 1716428
Bureau balance samples: 27299925


In [15]:
brDf.set_index('SK_ID_BUREAU', drop=True, inplace=True)
print('Index of bureau set to SK_ID_BUREAU, check unicity:',
      brDf.index.is_unique)

Index of bureau set to SK_ID_BUREAU, check unicity: True


In [16]:
describe_categorical(brDf, True)

,count,missing freq,unique,mode,mode count,mode freq,value samples
CREDIT_ACTIVE,1716428,0.0,4,Closed,1079273,62.879,"[Closed, Active, Sold, Bad debt]"
CREDIT_CURRENCY,1716428,0.0,4,currency 1,1715020,99.918,"[currency 1, currency 2, currency 4, currency 3]"
CREDIT_TYPE,1716428,0.0,15,Consumer credit,1251615,72.920,"[Consumer credit, Credit card, Mortgage, Car loan, Microloan, Loan for working capital replenishment, Loan for business development, Real estate loan, Unknown type of loan, Another type of loan]"


In [17]:
describe_int(brDf, True)

,count,missing freq,unique,mode,mode count,mode freq,mean,std,min,25%,50%,75%,max
DAYS_ENDDATE_FACT,1082775,36.917,2917,-329,811,0.075,-1017.437148,714.010626,-42023,-1489,-897,-425,0
DAYS_CREDIT_ENDDATE,1610875,6.150,14096,0,883,0.055,510.517362,4994.219837,-42060,-1138,-330,474,31199
DAYS_CREDIT,1716428,0.000,2923,-364,1330,0.077,-1142.107685,795.164928,-2922,-1666,-987,-474,0
SK_ID_CURR,1716428,0.000,305811,120860,116,0.007,278214.933645,102938.558112,100001,188866,278055,367426,456255
CREDIT_DAY_OVERDUE,1716428,0.000,942,0,1712211,99.754,0.818167,36.544428,0,0,0,0,2792
CNT_CREDIT_PROLONG,1716428,0.000,10,0,1707314,99.469,0.006410,0.096224,0,0,0,0,9
DAYS_CREDIT_UPDATE,1716428,0.000,2982,-7,18503,1.078,-593.748320,720.747312,-41947,-908,-395,-33,372


In [18]:
describe_float(brDf, True)

,count,missing freq,mean,std,min,25%,50%,75%,max
AMT_ANNUITY,489637,71.473,15712.757700,3.258269e+05,0.000,0.0,0.0,13500.0,1.184534e+08
AMT_CREDIT_MAX_OVERDUE,591940,65.513,3825.417661,2.060316e+05,0.000,0.0,0.0,0.0,1.159872e+08
AMT_CREDIT_SUM_LIMIT,1124648,34.477,6229.514980,4.503203e+04,-586406.115,0.0,0.0,0.0,4.705600e+06
AMT_CREDIT_SUM_DEBT,1458759,15.012,137085.119952,6.774011e+05,-4705600.320,0.0,0.0,40153.5,1.701000e+08
AMT_CREDIT_SUM,1716415,0.001,354994.591918,1.149811e+06,0.000,51300.0,125518.5,315000.0,5.850000e+08
AMT_CREDIT_SUM_OVERDUE,1716428,0.000,37.912758,5.937650e+03,0.000,0.0,0.0,0.0,3.756681e+06


In [19]:
describe_categorical(bbDf, True)

,count,missing freq,unique,mode,mode count,mode freq,value samples
STATUS,27299925,0.0,8,C,13646993,49.989,"[C, 0, X, 1, 2, 3, 5, 4]"


In [20]:
describe_int(bbDf, True)

,count,missing freq,unique,mode,mode count,mode freq,mean,std,min,25%,50%,75%,max
SK_ID_BUREAU,27299925,0.0,817395,5001709,97,0.000,6.036297e+06,492348.856904,5001709,5730933,6070821,6431951,6842888
MONTHS_BALANCE,27299925,0.0,97,-1,622601,2.281,-3.074169e+01,23.864509,-96,-46,-25,-11,0


In [21]:
describe_float(bbDf, True)

,count,missing freq,mean,std,min,25%,50%,75%,max


#### Nettoyage & encodage catégoriel

In [22]:
def bureau_cleaning_encoding(df,
                             context=None,
                             verbose=True):

    # Local overriding of print
    def _print(*args, **kwargs):
        if 'tabs' in kwargs:
            tabs = int(kwargs['tabs'])
            del kwargs['tabs']
        else:
            tabs = 2
        if verbose:
            print('\t'*tabs, *args, **kwargs)
        else:
            pass

    _print('Cleaning & encoding of bureau', tabs=1)

    # Potentially create & fit the encoders for the categorical features
    if context is None:
        context = {}
    # Encoder for multi-class categorical features of bureau
    if 'bureau_multi_cat_features_encoder' not in context:
        desc = describe_categorical(df)
        # Create & fit a one hot encoder => encoding to 0, 1 onto several cols
        encoder = OneHotEncoder(drop=None,
                                handle_unknown='error',
                                sparse_output=False,
                                dtype=np.float64,
                                feature_name_combiner=one_hot_name_combiner)
        encoder.fit(df[desc.index.to_list()])
        context['bureau_multi_cat_features_encoder'] = encoder

    # Encoding multi-class categorical features of bureau (n features with
    #  value 0 or 1)
    # Note that missing values are one-hot encoded in specific features
    encoder = context['bureau_multi_cat_features_encoder']
    tmpData = encoder.transform(df[encoder.feature_names_in_])
    df.loc[:, encoder.get_feature_names_out()] = tmpData

    # Defragmenting the DataFrame
    df = df.copy()

    return df, context


def bureau_balance_cleaning_encoding(df,
                                     context=None,
                                     verbose=True):

    # Local overriding of print
    def _print(*args, **kwargs):
        if 'tabs' in kwargs:
            tabs = int(kwargs['tabs'])
            del kwargs['tabs']
        else:
            tabs = 2
        if verbose:
            print('\t'*tabs, *args, **kwargs)
        else:
            pass

    _print('Cleaning & encoding of bureau_balance', tabs=1)

    # Potentially create & fit the encoders for the categorical features
    if context is None:
        context = {}
    # Encoder for multi-class categorical features of bureau_balance
    if 'bureau_bal_multi_cat_features_encoder' not in context:
        desc = describe_categorical(df)
        # Create & fit a one hot encoder => encoding to 0, 1 onto several cols
        encoder = OneHotEncoder(drop=None,
                                handle_unknown='error',
                                sparse_output=False,
                                dtype=np.float64,
                                feature_name_combiner=one_hot_name_combiner)
        encoder.fit(df[desc.index.to_list()])
        context['bureau_bal_multi_cat_features_encoder'] = encoder

    # Encoding multi-class categorical features of bureau_balance (n features
    # with value 0 or 1)
    # Note that missing values are one-hot encoded in specific features
    encoder = context['bureau_bal_multi_cat_features_encoder']
    tmpData = encoder.transform(df[encoder.feature_names_in_])
    df.loc[:, encoder.get_feature_names_out()] = tmpData

    # Defragmenting the DataFrame
    df = df.copy()

    return df, context

#### Imputation

In [23]:
def bureau_imputation(df,
                      relatedDfs,
                      context=None,
                      verbose=True):

    # Local overriding of print
    def _print(*args, **kwargs):
        if 'tabs' in kwargs:
            tabs = int(kwargs['tabs'])
            del kwargs['tabs']
        else:
            tabs = 2
        if verbose:
            print('\t'*tabs, *args, **kwargs)
        else:
            pass

    _print('No imputation for bureau', tabs=1)

    return df, context


def bureau_balance_imputation(df,
                              relatedDfs,
                              context=None,
                              verbose=True):

    # Local overriding of print
    def _print(*args, **kwargs):
        if 'tabs' in kwargs:
            tabs = int(kwargs['tabs'])
            del kwargs['tabs']
        else:
            tabs = 2
        if verbose:
            print('\t'*tabs, *args, **kwargs)
        else:
            pass

    _print('No imputation for bureau_balance', tabs=1)

    return df, context

#### Feature engineering

In [24]:
def bureau_feature_engineering(df,
                               relatedDfs,
                               context=None,
                               verbose=True):

    # Local overriding of print
    def _print(*args, **kwargs):
        if 'tabs' in kwargs:
            tabs = int(kwargs['tabs'])
            del kwargs['tabs']
        else:
            tabs = 2
        if verbose:
            print('\t'*tabs, *args, **kwargs)
        else:
            pass

    _print('No feature engineering for bureau', tabs=1)

    return df, context


def bureau_balance_feature_engineering(df,
                                       relatedDfs,
                                       context=None,
                                       verbose=True):

    # Local overriding of print
    def _print(*args, **kwargs):
        if 'tabs' in kwargs:
            tabs = int(kwargs['tabs'])
            del kwargs['tabs']
        else:
            tabs = 2
        if verbose:
            print('\t'*tabs, *args, **kwargs)
        else:
            pass

    _print('No feature engineering for bureau_balance', tabs=1)

    return df, context

#### Jointure & feature engineering d'agrégation de bureau & bureau_balance

In [25]:
def bureau_balance_bureau_merging(mainDf,
                                  secDf,
                                  context=None,
                                  verbose=True):

    # Local overriding of print
    def _print(*args, **kwargs):
        if 'tabs' in kwargs:
            tabs = int(kwargs['tabs'])
            del kwargs['tabs']
        else:
            tabs = 2
        if verbose:
            print('\t'*tabs, *args, **kwargs)
        else:
            pass

    _print('Merging bureau & bureau_balance:',
           'aggregation feature engineering',
           tabs=1)

    # Aggregation w/ feature engineering
    aggRules = {}
    aggRules['MONTHS_BALANCE'] = ['min', 'max', 'size']
    for col in [col for col in secDf.columns if col.startswith('ONEHOT__')]:
        aggRules[col] = ['mean']
    grpdSecDf = secDf.groupby('SK_ID_BUREAU').agg(aggRules)
    # Renaming aggregated columns before merging
    cols = [e[0] + '__BUROBAL_' + e[1].lower() for e in grpdSecDf.columns]
    grpdSecDf.columns = pd.Index(cols)

    # Merging
    mainDf = pd.merge(mainDf,
                      grpdSecDf,
                      how='left',
                      left_index=True,
                      right_index=True)

    # Defragmenting the DataFrame
    df = mainDf.copy()

    return df, context

In [26]:
def bureau_application_merging(mainDf,
                               secDf,
                               context=None,
                               verbose=True):

    # Local overriding of print
    def _print(*args, **kwargs):
        if 'tabs' in kwargs:
            tabs = int(kwargs['tabs'])
            del kwargs['tabs']
        else:
            tabs = 2
        if verbose:
            print('\t'*tabs, *args, **kwargs)
        else:
            pass

    _print('Merging application & bureau:',
           'aggregation feature engineering',
           tabs=1)

    # Aggregation w/ feature engineering
    numAggRules = {}
    numAggRules['DAYS_CREDIT'] = ['min', 'max', 'mean', 'var']
    numAggRules['DAYS_CREDIT_ENDDATE'] = ['min', 'max', 'mean']
    numAggRules['DAYS_CREDIT_UPDATE'] = ['mean']
    numAggRules['CREDIT_DAY_OVERDUE'] = ['max', 'mean']
    numAggRules['AMT_CREDIT_MAX_OVERDUE'] = ['mean']
    numAggRules['AMT_CREDIT_SUM'] = ['max', 'mean', 'sum']
    numAggRules['AMT_CREDIT_SUM_DEBT'] = ['max', 'mean', 'sum']
    numAggRules['AMT_CREDIT_SUM_OVERDUE'] = ['mean']
    numAggRules['AMT_CREDIT_SUM_LIMIT'] = ['mean', 'sum']
    numAggRules['AMT_ANNUITY'] = ['max', 'mean']
    numAggRules['CNT_CREDIT_PROLONG'] = ['sum']
    numAggRules['MONTHS_BALANCE__BUROBAL_min'] = ['min']
    numAggRules['MONTHS_BALANCE__BUROBAL_max'] = ['max']
    numAggRules['MONTHS_BALANCE__BUROBAL_size'] = ['mean', 'sum']
    catAggRules = {}
    for col in [col for col in secDf.columns if col.startswith('ONEHOT__')]:
        catAggRules[col] = ['mean']
    grpdSecDf = secDf.groupby('SK_ID_CURR').agg({**numAggRules, **catAggRules})
    # Renaming aggregated columns before merging
    cols = [e[0] + '__BURO_' + e[1].lower() for e in grpdSecDf.columns]
    grpdSecDf.columns = pd.Index(cols)

    # Bureau: Active credits - using only numerical aggregations
    activeCredDf = secDf[secDf['ONEHOT__CREDIT_ACTIVE__Active'] == 1]
    grpdActDf = activeCredDf.groupby('SK_ID_CURR').agg(numAggRules)
    # Renaming aggregated columns before merging
    cols = [e[0] + '__BURO_active_' + e[1].lower() for e in grpdActDf.columns]
    grpdActDf.columns = pd.Index(cols)

    # Bureau: Closed credits - using only numerical aggregations
    closedCredDf = secDf[secDf['ONEHOT__CREDIT_ACTIVE__Closed'] == 1]
    grpdClsdDf = closedCredDf.groupby('SK_ID_CURR').agg(numAggRules)
    # Renaming aggregated columns before merging
    cols = [e[0] + '__BURO_closed_' + e[1].lower() for e in grpdClsdDf.columns]
    grpdClsdDf.columns = pd.Index(cols)

    # Merging
    grpdSecDf = pd.merge(grpdSecDf,
                         grpdActDf,
                         how='left',
                         left_index=True,
                         right_index=True)
    grpdSecDf = pd.merge(grpdSecDf,
                         grpdClsdDf,
                         how='left',
                         left_index=True,
                         right_index=True)
    mainDf = pd.merge(mainDf,
                      grpdSecDf,
                      how='left',
                      left_index=True,
                      right_index=True)

    # Defragmenting the DataFrame
    df = mainDf.copy()

    return df, context

#### Traitement

In [27]:
with timer('Process bureau & bureau_balance'):
    brDf, context = bureau_cleaning_encoding(brDf,
                                             context,
                                             True)
    bbDf, context = bureau_balance_cleaning_encoding(bbDf,
                                                     context,
                                                     True)
    brDf, context = bureau_imputation(brDf,
                                      {'application': apDf},
                                      context,
                                      True)
    bbDf, context = bureau_balance_imputation(bbDf,
                                              {
                                                  'application': apDf,
                                                  'bureau': brDf,
                                              },
                                              context,
                                              True)
    brDf, context = bureau_feature_engineering(brDf,
                                               {'application': apDf},
                                               context,
                                               True)
    bbDf, context = bureau_balance_feature_engineering(bbDf,
                                                       {
                                                           'application': apDf,
                                                           'bureau': brDf,
                                                       },
                                                       context,
                                                       True)
    brDf, context = bureau_balance_bureau_merging(brDf,
                                                  bbDf,
                                                  context,
                                                  True)
    apDf, context = bureau_application_merging(apDf,
                                               brDf,
                                               context,
                                               True)
    # Freeing memory
    bbDf = None
    brDf = None
    gc.collect()
print('DataFrame shape:', apDf.shape)

Process bureau & bureau_balance - starting
	 Cleaning & encoding of bureau
	 Cleaning & encoding of bureau_balance
	 No imputation for bureau
	 No imputation for bureau_balance
	 No feature engineering for bureau
	 No feature engineering for bureau_balance
	 Merging bureau & bureau_balance: aggregation feature engineering
	 Merging application & bureau: aggregation feature engineering
Process bureau & bureau_balance - done in 39s
DataFrame shape: (356251, 377)


### Fichier previous application

In [28]:
papDf = pd.read_csv(r'data\previous_application.csv', low_memory=False)
print('Previous application samples: {}'.format(len(papDf)))

Previous application samples: 1670214


In [29]:
papDf.set_index('SK_ID_PREV', drop=True, inplace=True)
print('Index set to SK_ID_PREV, check unicity:', papDf.index.is_unique)

Index set to SK_ID_PREV, check unicity: True


In [30]:
describe_categorical(papDf, True)

,count,missing freq,unique,mode,mode count,mode freq,value samples
NAME_TYPE_SUITE,849809,49.120,7,Unaccompanied,508970,59.892,"[Unaccompanied, Spouse, partner, Family, Children, Other_B, Other_A, Group of people]"
PRODUCT_COMBINATION,1669868,0.021,17,Cash,285990,17.127,"[POS mobile with interest, Cash X-Sell: low, Cash X-Sell: high, Cash X-Sell: middle, Cash Street: high, Cash, POS household without interest, POS household with interest, POS other with interest, Card X-Sell]"
NAME_CONTRACT_TYPE,1670214,0.000,4,Cash loans,747553,44.758,"[Consumer loans, Cash loans, Revolving loans, XNA]"
WEEKDAY_APPR_PROCESS_START,1670214,0.000,7,TUESDAY,255118,15.275,"[SATURDAY, THURSDAY, TUESDAY, MONDAY, FRIDAY, SUNDAY, WEDNESDAY]"
FLAG_LAST_APPL_PER_CONTRACT,1670214,0.000,2,Y,1661739,99.493,"[Y, N]"
NAME_CASH_LOAN_PURPOSE,1670214,0.000,25,XAP,922661,55.242,"[XAP, XNA, Repairs, Everyday expenses, Car repairs, Building a house or an annex, Other, Journey, Purchase of electronic equipment, Medicine]"
NAME_CONTRACT_STATUS,1670214,0.000,4,Approved,1036781,62.075,"[Approved, Refused, Canceled, Unused offer]"
NAME_PAYMENT_TYPE,1670214,0.000,4,Cash through the bank,1033552,61.881,"[Cash through the bank, XNA, Non-cash from your account, Cashless from the account of the employer]"
CODE_REJECT_REASON,1670214,0.000,9,XAP,1353093,81.013,"[XAP, HC, LIMIT, CLIENT, SCOFR, SCO, XNA, VERIF, SYSTEM]"
NAME_CLIENT_TYPE,1670214,0.000,4,Repeater,1231261,73.719,"[Repeater, New, Refreshed, XNA]"


In [31]:
describe_int(papDf, True)

,count,missing freq,unique,mode,mode count,mode freq,mean,std,min,25%,50%,75%,max
DAYS_FIRST_DRAWING,997149,40.298,2838,365243,934444,93.712,342209.855039,88916.115833,-2922,365243,365243,365243,365243
DAYS_FIRST_DUE,997149,40.298,2892,365243,40645,4.076,13826.269337,72444.869708,-2892,-1628,-831,-411,365243
DAYS_LAST_DUE,997149,40.298,2873,365243,211221,21.182,76582.403064,149647.415123,-2889,-1314,-537,-74,365243
DAYS_LAST_DUE_1ST_VERSION,997149,40.298,4605,365243,93864,9.413,33767.774054,106857.034789,-2801,-1242,-361,129,365243
NFLAG_INSURED_ON_APPROVAL,997149,40.298,2,0,665527,66.743,0.332570,0.471134,0,0,0,1,1
DAYS_TERMINATION,997149,40.298,2830,365243,225913,22.656,81992.343838,153303.516729,-2874,-1270,-499,-44,365243
CNT_PAYMENT,1297984,22.286,49,12,323049,24.889,16.054082,14.567288,0,6,12,24,84
SK_ID_CURR,1670214,0.000,338857,187868,77,0.005,278357.174099,102814.823849,100001,189329,278714,367514,456255
SELLERPLACE_AREA,1670214,0.000,2097,-1,762675,45.663,313.951115,7127.443459,-1,-1,3,82,4000000
HOUR_APPR_PROCESS_START,1670214,0.000,24,11,192728,11.539,12.484182,3.334028,0,10,12,15,23


In [32]:
describe_float(papDf, True)

,count,missing freq,mean,std,min,25%,50%,75%,max
RATE_INTEREST_PRIMARY,5951,99.644,0.188357,0.087671,0.034781,0.160716,0.189122,0.193330,1.000
RATE_INTEREST_PRIVILEGED,5951,99.644,0.773503,0.100879,0.373150,0.715645,0.835095,0.852537,1.000
AMT_DOWN_PAYMENT,774370,53.636,6697.402139,20921.495410,-0.900000,0.000000,1638.000000,7740.000000,3060045.000
RATE_DOWN_PAYMENT,774370,53.636,0.079637,0.107823,-0.000015,0.000000,0.051605,0.108909,1.000
AMT_GOODS_PRICE,1284699,23.082,227847.279283,315396.557937,0.000000,50841.000000,112320.000000,234000.000000,6905160.000
AMT_ANNUITY,1297979,22.287,15955.120659,14782.137335,0.000000,6321.780000,11250.000000,20658.420000,418058.145
AMT_CREDIT,1670213,0.000,196114.021218,318574.616547,0.000000,24160.500000,80541.000000,216418.500000,6905160.000
AMT_APPLICATION,1670214,0.000,175233.860360,292779.762386,0.000000,18720.000000,71046.000000,180360.000000,6905160.000


#### Nettoyage & encodage catégoriel

In [33]:
def prevapplication_cleaning_encoding(df,
                                      context=None,
                                      verbose=True):

    # Local overriding of print
    def _print(*args, **kwargs):
        if 'tabs' in kwargs:
            tabs = int(kwargs['tabs'])
            del kwargs['tabs']
        else:
            tabs = 2
        if verbose:
            print('\t'*tabs, *args, **kwargs)
        else:
            pass

    _print('Cleaning & encoding of previous_application', tabs=1)

    # NaN values for DAYS_FIRST_DRAWING which value is 365243 (outlier)
    col = 'DAYS_FIRST_DRAWING'
    cnt = df.loc[df[col] == 365243, :].shape[0]
    _print('Number of applications with DAYS_FIRST_DRAWING=365243:',
           cnt)
    _print(f'Updating {cnt} application(s),',
           'setting DAYS_FIRST_DRAWING from 365243 to NaN')
    df.loc[df[col] == 365243, col] = np.nan
    # NaN values for DAYS_FIRST_DUE which value is 365243 (outlier)
    col = 'DAYS_FIRST_DUE'
    cnt = df.loc[df[col] == 365243, :].shape[0]
    _print('Number of applications with DAYS_FIRST_DUE=365243:',
           cnt)
    _print(f'Updating {cnt} application(s),',
           'setting DAYS_FIRST_DUE from 365243 to NaN')
    df.loc[df[col] == 365243, col] = np.nan
    # NaN values for DAYS_LAST_DUE_1ST_VERSION which value is 365243 (outlier)
    col = 'DAYS_LAST_DUE_1ST_VERSION'
    cnt = df.loc[df[col] == 365243, :].shape[0]
    _print('Number of applications with DAYS_LAST_DUE_1ST_VERSION=365243:',
           cnt)
    _print(f'Updating {cnt} application(s),',
           'setting DAYS_LAST_DUE_1ST_VERSION from 365243 to NaN')
    df.loc[df[col] == 365243, col] = np.nan
    # NaN values for DAYS_LAST_DUE which value is 365243 (outlier)
    col = 'DAYS_LAST_DUE'
    cnt = df.loc[df[col] == 365243, :].shape[0]
    _print('Number of applications with DAYS_LAST_DUE=365243:',
           cnt)
    _print(f'Updating {cnt} application(s),',
           'setting DAYS_LAST_DUE from 365243 to NaN')
    df.loc[df[col] == 365243, col] = np.nan
    # NaN values for DAYS_TERMINATION which value is 365243 (outlier)
    col = 'DAYS_TERMINATION'
    cnt = df.loc[df[col] == 365243, :].shape[0]
    _print('Number of applications with DAYS_TERMINATION=365243:',
           cnt)
    _print(f'Updating {cnt} application(s),',
           'setting DAYS_TERMINATION from 365243 to NaN')
    df.loc[df[col] == 365243, col] = np.nan

    # Potentially create & fit the encoders for the categorical features
    if context is None:
        context = {}
        desc = None
    # Encoder for 2-class categorical features
    if 'prevapp_bin_cat_features_encoder' not in context:
        # Get a description of the categorical features
        desc = describe_categorical(df)
        # List of binary categorical features
        binCatCols = desc[desc['unique'] == 2].index.to_list()
        # Create & fit an ordinal encoder => encoding to 0, 1 or NaN
        encoder = OrdinalEncoder(handle_unknown='error',
                                 dtype=np.float64).fit(df[binCatCols])
        context['prevapp_bin_cat_features_encoder'] = encoder
    # Encoder for multi-class categorical features
    if 'prevapp_multi_cat_features_encoder' not in context:
        if desc is None:
            # Get a description of the categorical features
            desc = describe_categorical(df)
        # List of multi-class categorical features
        multiCatCols = desc[desc['unique'] > 2].index.to_list()
        # Create & fit a one hot encoder => encoding to 0, 1 onto several cols
        encoder = OneHotEncoder(drop=None,
                                handle_unknown='error',
                                sparse_output=False,
                                dtype=np.float64,
                                feature_name_combiner=one_hot_name_combiner)
        encoder.fit(df[multiCatCols])
        context['prevapp_multi_cat_features_encoder'] = encoder

    # Encoding binary categorical features (0, 1 or NaN)
    encoder = context['prevapp_bin_cat_features_encoder']
    tmpData = encoder.transform(df[encoder.feature_names_in_])
    df.loc[:, 'BINARY__' + encoder.feature_names_in_] = tmpData

    # Encoding multi-class categorical features (n features with value 0, 1)
    # Note that missing values are one-hot encoded in specific features
    encoder = context['prevapp_multi_cat_features_encoder']
    tmpData = encoder.transform(df[encoder.feature_names_in_])
    df.loc[:, encoder.get_feature_names_out()] = tmpData

    # Defragmenting the DataFrame
    df = df.copy()

    return df, context

#### Imputation

In [34]:
def prevapplication_imputation(df,
                               relatedDfs,
                               context=None,
                               verbose=True):

    # Local overriding of print
    def _print(*args, **kwargs):
        if 'tabs' in kwargs:
            tabs = int(kwargs['tabs'])
            del kwargs['tabs']
        else:
            tabs = 2
        if verbose:
            print('\t'*tabs, *args, **kwargs)
        else:
            pass

    _print('No imputation for previous_application', tabs=1)

    return df, context

#### Feature engineering

In [35]:
def prevapplication_feature_engineering(df,
                                        relatedDfs,
                                        context=None,
                                        verbose=True):

    # Local overriding of print
    def _print(*args, **kwargs):
        if 'tabs' in kwargs:
            tabs = int(kwargs['tabs'])
            del kwargs['tabs']
        else:
            tabs = 2
        if verbose:
            print('\t'*tabs, *args, **kwargs)
        else:
            pass

    _print('Feature engineering for previous_application', tabs=1)

    # Some simple new features
    _print('Creating feature: value ask / value received ratio in %')
    df['APP_CREDIT_PERC'] = df['AMT_APPLICATION'] / df['AMT_CREDIT']
    # If we have divided by 0
    df.loc[df['APP_CREDIT_PERC'] == np.inf, 'APP_CREDIT_PERC'] = np.nan

    # Defragmenting the DataFrame
    df = df.copy()

    return df, context

#### Jointure & feature engineering d'agrégation de bureau & bureau_balance

In [36]:
def prevapplication_application_merging(mainDf,
                                        secDf,
                                        context=None,
                                        verbose=True):

    # Local overriding of print
    def _print(*args, **kwargs):
        if 'tabs' in kwargs:
            tabs = int(kwargs['tabs'])
            del kwargs['tabs']
        else:
            tabs = 2
        if verbose:
            print('\t'*tabs, *args, **kwargs)
        else:
            pass

    _print('Merging application & previous_application:',
           'aggregation feature engineering',
           tabs=1)

    # Aggregation w/ feature engineering
    numAggRules = {}
    numAggRules['AMT_ANNUITY'] = ['min', 'max', 'mean']
    numAggRules['AMT_APPLICATION'] = ['min', 'max', 'mean']
    numAggRules['AMT_CREDIT'] = ['min', 'max', 'mean']
    numAggRules['APP_CREDIT_PERC'] = ['min', 'max', 'mean', 'var']
    numAggRules['AMT_DOWN_PAYMENT'] = ['min', 'max', 'mean']
    numAggRules['AMT_GOODS_PRICE'] = ['min', 'max', 'mean']
    numAggRules['HOUR_APPR_PROCESS_START'] = ['min', 'max', 'mean']
    numAggRules['RATE_DOWN_PAYMENT'] = ['min', 'max', 'mean']
    numAggRules['DAYS_DECISION'] = ['min', 'max', 'mean']
    numAggRules['CNT_PAYMENT'] = ['mean', 'sum']
    catAggRules = {}
    for col in [col for col in secDf.columns if col.startswith('BINARY__')]:
        catAggRules[col] = ['mean']
    for col in [col for col in secDf.columns if col.startswith('ONEHOT__')]:
        catAggRules[col] = ['mean']
    grpdSecDf = secDf.groupby('SK_ID_CURR').agg({**numAggRules, **catAggRules})
    # Renaming aggregated columns before merging
    cols = [e[0] + '__PRVAPP_' + e[1].lower() for e in grpdSecDf.columns]
    grpdSecDf.columns = pd.Index(cols)

    # Previous applications: approved application - only numerical aggregations
    activeCredDf = secDf[secDf['ONEHOT__NAME_CONTRACT_STATUS__Approved'] == 1]
    grpdAprDf = activeCredDf.groupby('SK_ID_CURR').agg(numAggRules)
    # Renaming aggregated columns before merging
    cols = [e[0] + '__PRVAPP_aprvd_' + e[1].lower() for e in grpdAprDf.columns]
    grpdAprDf.columns = pd.Index(cols)

    # Previous applications: refused application - only numerical aggregations
    closedCredDf = secDf[secDf['ONEHOT__NAME_CONTRACT_STATUS__Refused'] == 1]
    grpdRfsDf = closedCredDf.groupby('SK_ID_CURR').agg(numAggRules)
    # Renaming aggregated columns before merging
    cols = [e[0] + '__PRVAPP_rfusd_' + e[1].lower() for e in grpdRfsDf.columns]
    grpdRfsDf.columns = pd.Index(cols)

    # Merging
    grpdSecDf = pd.merge(grpdSecDf,
                         grpdAprDf,
                         how='left',
                         left_index=True,
                         right_index=True)
    grpdSecDf = pd.merge(grpdSecDf,
                         grpdRfsDf,
                         how='left',
                         left_index=True,
                         right_index=True)
    mainDf = pd.merge(mainDf,
                      grpdSecDf,
                      how='left',
                      left_index=True,
                      right_index=True)

    # Defragmenting the DataFrame
    df = mainDf.copy()

    return df, context

#### Traitement

In [37]:
with timer('Process previous_application'):
    papDf, context = prevapplication_cleaning_encoding(papDf,
                                                       context,
                                                       True)
    papDf, context = prevapplication_imputation(papDf,
                                                {'application': apDf},
                                                context,
                                                True)
    papDf, context = prevapplication_feature_engineering(papDf,
                                                         {'application': apDf},
                                                         context,
                                                         True)
    apDf, context = prevapplication_application_merging(apDf,
                                                        papDf,
                                                        context,
                                                        True)
    gc.collect()
print('DataFrame shape:', apDf.shape)

Process previous_application - starting
	 Cleaning & encoding of previous_application
		 Number of applications with DAYS_FIRST_DRAWING=365243: 934444
		 Updating 934444 application(s), setting DAYS_FIRST_DRAWING from 365243 to NaN
		 Number of applications with DAYS_FIRST_DUE=365243: 40645
		 Updating 40645 application(s), setting DAYS_FIRST_DUE from 365243 to NaN
		 Number of applications with DAYS_LAST_DUE_1ST_VERSION=365243: 93864
		 Updating 93864 application(s), setting DAYS_LAST_DUE_1ST_VERSION from 365243 to NaN
		 Number of applications with DAYS_LAST_DUE=365243: 211221
		 Updating 211221 application(s), setting DAYS_LAST_DUE from 365243 to NaN
		 Number of applications with DAYS_TERMINATION=365243: 225913
		 Updating 225913 application(s), setting DAYS_TERMINATION from 365243 to NaN
	 No imputation for previous_application
	 Feature engineering for previous_application
		 Creating feature: value ask / value received ratio in %
	 Merging application & previous_application: agg

### Fichier installments_payments

In [38]:
ipDf = pd.read_csv(r'data\installments_payments.csv', low_memory=False)
print('Installments payments samples: {}'.format(len(ipDf)))

Installments payments samples: 13605401


In [39]:
describe_int(ipDf, True)

,count,missing freq,unique,mode,mode count,mode freq,mean,std,min,25%,50%,75%,max
DAYS_ENTRY_PAYMENT,13602496,0.021,3039,-91,13103,0.096,-1.051114e+03,800.585883,-4921,-1662,-827,-370,-1
SK_ID_PREV,13605401,0.000,997752,2360056,293,0.002,1.903365e+06,536202.905546,1000001,1434191,1896520,2369094,2843499
NUM_INSTALMENT_VERSION,13605401,0.000,65,1,8485004,62.365,8.566373e-01,1.035216,0,0,1,1,178
SK_ID_CURR,13605401,0.000,339587,145728,372,0.003,2.784449e+05,102718.310411,100001,189639,278685,367530,456255
NUM_INSTALMENT_NUMBER,13605401,0.000,277,1,1004160,7.381,1.887090e+01,26.664067,1,4,8,19,277
DAYS_INSTALMENT,13605401,0.000,2922,-120,11512,0.085,-1.042270e+03,800.946284,-2922,-1654,-818,-361,-1


In [40]:
describe_float(ipDf, True)

,count,missing freq,mean,std,min,25%,50%,75%,max
AMT_PAYMENT,13602496,0.021,17238.223250,54735.783981,0.0,3398.265,8125.515,16108.425,3771487.845
AMT_INSTALMENT,13605401,0.000,17050.906989,50570.254429,0.0,4226.085,8884.080,16710.210,3771487.845


#### Nettoyage & encodage catégoriel

In [41]:
def installments_cleaning_encoding(df,
                                   context=None,
                                   verbose=True):

    # Local overriding of print
    def _print(*args, **kwargs):
        if 'tabs' in kwargs:
            tabs = int(kwargs['tabs'])
            del kwargs['tabs']
        else:
            tabs = 2
        if verbose:
            print('\t'*tabs, *args, **kwargs)
        else:
            pass

    _print('No cleaning or encoding of installments_payments', tabs=1)

    return df, context

#### Imputation

In [42]:
def installments_imputation(df,
                            relatedDfs,
                            context=None,
                            verbose=True):

    # Local overriding of print
    def _print(*args, **kwargs):
        if 'tabs' in kwargs:
            tabs = int(kwargs['tabs'])
            del kwargs['tabs']
        else:
            tabs = 2
        if verbose:
            print('\t'*tabs, *args, **kwargs)
        else:
            pass

    _print('No imputation for installments_payments', tabs=1)

    return df, context

#### Feature engineering

In [43]:
def installments_feature_engineering(df,
                                     relatedDfs,
                                     context=None,
                                     verbose=True):

    # Local overriding of print
    def _print(*args, **kwargs):
        if 'tabs' in kwargs:
            tabs = int(kwargs['tabs'])
            del kwargs['tabs']
        else:
            tabs = 2
        if verbose:
            print('\t'*tabs, *args, **kwargs)
        else:
            pass

    _print('Feature engineering for installments_payments', tabs=1)

    # Some simple new features
    _print('Creating feature: payment / installment ratio in %')
    df['PAYMENT_PERC'] = df['AMT_PAYMENT'] / df['AMT_INSTALMENT']
    # If we have divided by 0
    df.loc[df['PAYMENT_PERC'] == np.inf, 'PAYMENT_PERC'] = np.nan
    _print('Creating feature: difference between installment and payment')
    df['PAYMENT_DIFF'] = df['AMT_INSTALMENT'] - df['AMT_PAYMENT']
    _print('Creating feature: Days Past Due')
    df['DPD'] = df['DAYS_ENTRY_PAYMENT'] - df['DAYS_INSTALMENT']
    df['DPD'] = df['DPD'].apply(lambda x: 0 if x < 0 else x)
    _print('Creating feature: Days Before Due')
    df['DBD'] = df['DAYS_INSTALMENT'] - df['DAYS_ENTRY_PAYMENT']
    df['DBD'] = df['DBD'].apply(lambda x: 0 if x < 0 else x)

    # Defragmenting the DataFrame
    df = df.copy()

    return df, context

#### Jointure & feature engineering d'agrégation de bureau & bureau_balance

In [44]:
def installments_application_merging(mainDf,
                                     secDf,
                                     context=None,
                                     verbose=True):

    # Local overriding of print
    def _print(*args, **kwargs):
        if 'tabs' in kwargs:
            tabs = int(kwargs['tabs'])
            del kwargs['tabs']
        else:
            tabs = 2
        if verbose:
            print('\t'*tabs, *args, **kwargs)
        else:
            pass

    _print('Merging installments_payments:',
           'aggregation feature engineering',
           tabs=1)

    # Aggregation w/ feature engineering
    aggRules = {}
    aggRules['NUM_INSTALMENT_VERSION'] = ['nunique']
    aggRules['DPD'] = ['max', 'mean', 'sum']
    aggRules['DBD'] = ['max', 'mean', 'sum']
    aggRules['PAYMENT_PERC'] = ['max', 'mean', 'sum', 'var']
    aggRules['PAYMENT_DIFF'] = ['max', 'mean', 'sum', 'var']
    aggRules['AMT_INSTALMENT'] = ['max', 'mean', 'sum']
    aggRules['AMT_PAYMENT'] = ['min', 'max', 'mean', 'sum']
    aggRules['DAYS_ENTRY_PAYMENT'] = ['max', 'mean', 'sum']

    secDfGrpby = secDf.groupby('SK_ID_CURR')
    grpdSecDf = secDfGrpby.agg(aggRules)
    # Renaming aggregated columns before merging
    cols = [e[0] + '__INSTAL_' + e[1].lower() for e in grpdSecDf.columns]
    grpdSecDf.columns = pd.Index(cols)
    grpdSecDf['INSTAL__INSTAL_count'] = secDfGrpby.size()

    # Merging
    mainDf = pd.merge(mainDf,
                      grpdSecDf,
                      how='left',
                      left_index=True,
                      right_index=True)

    # Defragmenting the DataFrame
    df = mainDf.copy()

    return df, context

#### Traitement

In [45]:
with timer('Process installments_payments'):
    ipDf, context = installments_cleaning_encoding(ipDf,
                                                   context,
                                                   True)
    ipDf, context = installments_imputation(ipDf,
                                            {
                                                'application': apDf,
                                                'prev_app': papDf,
                                            },
                                            context,
                                            True)
    ipDf, context = installments_feature_engineering(ipDf,
                                                     {
                                                         'application': apDf,
                                                         'prev_app': papDf,
                                                     },
                                                     context,
                                                     True)
    apDf, context = installments_application_merging(apDf,
                                                     ipDf,
                                                     context,
                                                     True)
    # Freeing memory
    ipDf = None
    gc.collect()
print('DataFrame shape:', apDf.shape)

Process installments_payments - starting
	 No cleaning or encoding of installments_payments
	 No imputation for installments_payments
	 Feature engineering for installments_payments
		 Creating feature: payment / installment ratio in %
		 Creating feature: difference between installment and payment
		 Creating feature: Days Past Due
		 Creating feature: Days Before Due
	 Merging installments_payments: aggregation feature engineering
Process installments_payments - done in 27s
DataFrame shape: (356251, 637)


### Fichier POS_CASH_balance

In [46]:
pcbDf = pd.read_csv(r'data\POS_CASH_balance.csv', low_memory=False)
print('POS CASH balance samples: {}'.format(len(pcbDf)))

POS CASH balance samples: 10001358


In [47]:
describe_categorical(pcbDf, True)

,count,missing freq,unique,mode,mode count,mode freq,value samples
NAME_CONTRACT_STATUS,10001358,0.0,9,Active,9151119,91.499,"[Active, Completed, Signed, Approved, Returned to the store, Demand, Canceled, XNA, Amortized debt]"


In [48]:
describe_int(pcbDf, True)

,count,missing freq,unique,mode,mode count,mode freq,mean,std,min,25%,50%,75%,max
CNT_INSTALMENT_FUTURE,9975271,0.261,79,0,1185960,11.889,1.048384e+01,11.109058,0,3,7,14,85
CNT_INSTALMENT,9975287,0.261,73,12,2496845,25.030,1.708965e+01,11.995056,1,10,12,24,92
SK_ID_CURR,10001358,0.000,337252,265042,295,0.003,2.784039e+05,102763.745090,100001,189550,278654,367429,456255
SK_ID_PREV,10001358,0.000,936325,1000256,96,0.001,1.903217e+06,535846.530722,1000001,1434405,1896565,2368963,2843499
MONTHS_BALANCE,10001358,0.000,96,-10,216441,2.164,-3.501259e+01,26.066570,-96,-54,-28,-13,-1
SK_DPD,10001358,0.000,3400,0,9706131,97.048,1.160693e+01,132.714043,0,0,0,0,4231
SK_DPD_DEF,10001358,0.000,2307,0,9887389,98.860,6.544684e-01,32.762491,0,0,0,0,3595


In [49]:
describe_float(pcbDf, True)

,count,missing freq,mean,std,min,25%,50%,75%,max


#### Nettoyage & encodage catégoriel

In [50]:
def pos_cash_cleaning_encoding(df,
                               context=None,
                               verbose=True):

    # Local overriding of print
    def _print(*args, **kwargs):
        if 'tabs' in kwargs:
            tabs = int(kwargs['tabs'])
            del kwargs['tabs']
        else:
            tabs = 2
        if verbose:
            print('\t'*tabs, *args, **kwargs)
        else:
            pass

    _print('Cleaning & encoding of POS_CASH_balance', tabs=1)

    # Potentially create & fit the encoders for the categorical features
    if context is None:
        context = {}
    # Encoder for multi-class categorical features
    if 'pos_cash_multi_cat_features_encoder' not in context:
        desc = describe_categorical(df)
        # Create & fit a one hot encoder => encoding to 0, 1 onto several cols
        encoder = OneHotEncoder(drop=None,
                                handle_unknown='error',
                                sparse_output=False,
                                dtype=np.float64,
                                feature_name_combiner=one_hot_name_combiner)
        encoder.fit(df[desc.index.to_list()])
        context['pos_cash_multi_cat_features_encoder'] = encoder

    # Encoding multi-class categorical features (n features with value 0, 1)
    # Note that missing values are one-hot encoded in specific features
    encoder = context['pos_cash_multi_cat_features_encoder']
    tmpData = encoder.transform(df[encoder.feature_names_in_])
    df.loc[:, encoder.get_feature_names_out()] = tmpData

    # Defragmenting the DataFrame
    df = df.copy()

    return df, context

#### Imputation

In [51]:
def pos_cash_imputation(df,
                        relatedDfs,
                        context=None,
                        verbose=True):

    # Local overriding of print
    def _print(*args, **kwargs):
        if 'tabs' in kwargs:
            tabs = int(kwargs['tabs'])
            del kwargs['tabs']
        else:
            tabs = 2
        if verbose:
            print('\t'*tabs, *args, **kwargs)
        else:
            pass

    _print('No imputation for POS_CASH_balance', tabs=1)

    return df, context

#### Feature engineering

In [52]:
def pos_cash_feature_engineering(df,
                                 relatedDfs,
                                 context=None,
                                 verbose=True):

    # Local overriding of print
    def _print(*args, **kwargs):
        if 'tabs' in kwargs:
            tabs = int(kwargs['tabs'])
            del kwargs['tabs']
        else:
            tabs = 2
        if verbose:
            print('\t'*tabs, *args, **kwargs)
        else:
            pass

    _print('No feature engineering for POS_CASH_balance', tabs=1)

    return df, context

#### Jointure & feature engineering d'agrégation de bureau & bureau_balance

In [53]:
def pos_cash_application_merging(mainDf,
                                 secDf,
                                 context=None,
                                 verbose=True):

    # Local overriding of print
    def _print(*args, **kwargs):
        if 'tabs' in kwargs:
            tabs = int(kwargs['tabs'])
            del kwargs['tabs']
        else:
            tabs = 2
        if verbose:
            print('\t'*tabs, *args, **kwargs)
        else:
            pass

    _print('Merging POS_CASH_balance:',
           'aggregation feature engineering',
           tabs=1)

    # Aggregation w/ feature engineering
    aggRules = {}
    aggRules['MONTHS_BALANCE'] = ['max', 'mean', 'size']
    aggRules['SK_DPD'] = ['max', 'mean']
    aggRules['SK_DPD_DEF'] = ['max', 'mean']
    for col in [col for col in secDf.columns if col.startswith('ONEHOT__')]:
        aggRules[col] = ['mean']

    secDfGrpby = secDf.groupby('SK_ID_CURR')
    grpdSecDf = secDfGrpby.agg(aggRules)
    # Renaming aggregated columns before merging
    cols = [e[0] + '__POSCASH_' + e[1].lower() for e in grpdSecDf.columns]
    grpdSecDf.columns = pd.Index(cols)
    grpdSecDf['POSCASH__POSCASH_count'] = secDfGrpby.size()

    # Merging
    mainDf = pd.merge(mainDf,
                      grpdSecDf,
                      how='left',
                      left_index=True,
                      right_index=True)

    # Defragmenting the DataFrame
    df = mainDf.copy()

    return df, context

#### Traitement

In [54]:
with timer('Process POS_CASH_balance'):
    pcbDf, context = pos_cash_cleaning_encoding(pcbDf,
                                                context,
                                                True)
    pcbDf, context = pos_cash_imputation(pcbDf,
                                         {
                                             'application': apDf,
                                             'prev_app': papDf,
                                         },
                                         context,
                                         True)
    pcbDf, context = pos_cash_feature_engineering(pcbDf,
                                                  {
                                                      'application': apDf,
                                                      'prev_app': papDf,
                                                  },
                                                  context,
                                                  True)
    apDf, context = pos_cash_application_merging(apDf,
                                                 pcbDf,
                                                 context,
                                                 True)
    # Freeing memory
    pcbDf = None
    gc.collect()
print('DataFrame shape:', apDf.shape)

Process POS_CASH_balance - starting
	 Cleaning & encoding of POS_CASH_balance
	 No imputation for POS_CASH_balance
	 No feature engineering for POS_CASH_balance
	 Merging POS_CASH_balance: aggregation feature engineering
Process POS_CASH_balance - done in 20s
DataFrame shape: (356251, 654)


### Fichier credit_card_balance

In [55]:
ccbDf = pd.read_csv(r'data\credit_card_balance.csv', low_memory=False)
print('Credit card balance samples: {}'.format(len(ccbDf)))

Credit card balance samples: 3840312


In [56]:
describe_categorical(ccbDf, True)

,count,missing freq,unique,mode,mode count,mode freq,value samples
NAME_CONTRACT_STATUS,3840312,0.0,7,Active,3698436,96.306,"[Active, Completed, Demand, Signed, Sent proposal, Refused, Approved]"


In [57]:
describe_int(ccbDf, True)

,count,missing freq,unique,mode,mode count,mode freq,mean,std,min,25%,50%,75%,max
CNT_DRAWINGS_OTHER_CURRENT,3090496,19.525,11,0,3077688,99.586,4.812496e-03,0.082639,0,0,0,0,12
CNT_DRAWINGS_POS_CURRENT,3090496,19.525,133,0,2825594,91.428,5.594791e-01,3.240649,0,0,0,0,165
CNT_DRAWINGS_ATM_CURRENT,3090496,19.525,44,0,2665718,86.255,3.094490e-01,1.100401,0,0,0,0,51
CNT_INSTALMENT_MATURE_CUM,3535076,7.948,121,0,551467,15.600,2.082508e+01,20.051494,0,4,15,32,120
AMT_CREDIT_LIMIT_ACTUAL,3840312,0.000,181,0,753823,19.629,1.538080e+05,165145.699525,0,45000,112500,180000,1350000
MONTHS_BALANCE,3840312,0.000,96,-4,102115,2.659,-3.452192e+01,26.667751,-96,-55,-28,-11,-1
SK_ID_CURR,3840312,0.000,103558,186401,192,0.005,2.783242e+05,102704.475133,100006,189517,278396,367580,456250
SK_ID_PREV,3840312,0.000,104307,1000096,96,0.002,1.904504e+06,536469.470563,1000018,1434385,1897122,2369327,2843496
CNT_DRAWINGS_CURRENT,3840312,0.000,129,0,3229952,84.106,7.031439e-01,3.190347,0,0,0,0,165
SK_DPD,3840312,0.000,917,0,3686957,96.007,9.283667e+00,97.515700,0,0,0,0,3260


In [58]:
describe_float(ccbDf, True)

,count,missing freq,mean,std,min,25%,50%,75%,max
AMT_PAYMENT_CURRENT,3072324,19.998,10280.537702,36078.084953,0.000,152.37,2702.7,9000.00000,4289207.445
AMT_DRAWINGS_ATM_CURRENT,3090496,19.525,5961.324822,28225.688578,-6827.310,0.00,0.0,0.00000,2115000.000
AMT_DRAWINGS_POS_CURRENT,3090496,19.525,2968.804848,20796.887047,0.000,0.00,0.0,0.00000,2239274.160
AMT_DRAWINGS_OTHER_CURRENT,3090496,19.525,288.169582,8201.989345,0.000,0.00,0.0,0.00000,1529847.000
AMT_INST_MIN_REGULARITY,3535076,7.948,3540.204129,5600.154122,0.000,0.00,0.0,6633.91125,202882.005
AMT_BALANCE,3840312,0.000,58300.155262,106307.031024,-420250.185,0.00,0.0,89046.68625,1505902.185
AMT_DRAWINGS_CURRENT,3840312,0.000,7433.388179,33846.077333,-6211.620,0.00,0.0,0.00000,2287098.315
AMT_PAYMENT_TOTAL_CURRENT,3840312,0.000,7588.856739,32005.987768,0.000,0.00,0.0,6750.00000,4278315.690
AMT_RECEIVABLE_PRINCIPAL,3840312,0.000,55965.876905,102533.616846,-423305.820,0.00,0.0,85359.24000,1472316.795
AMT_RECIVABLE,3840312,0.000,58088.811177,105965.369908,-420250.185,0.00,0.0,88899.49125,1493338.185


#### Nettoyage & encodage catégoriel

In [59]:
def credit_card_cleaning_encoding(df,
                                  context=None,
                                  verbose=True):

    # Local overriding of print
    def _print(*args, **kwargs):
        if 'tabs' in kwargs:
            tabs = int(kwargs['tabs'])
            del kwargs['tabs']
        else:
            tabs = 2
        if verbose:
            print('\t'*tabs, *args, **kwargs)
        else:
            pass

    _print('Cleaning & encoding of credit_card_balance', tabs=1)

    # Potentially create & fit the encoders for the categorical features
    if context is None:
        context = {}
    # Encoder for multi-class categorical features
    if 'credit_card_multi_cat_features_encoder' not in context:
        desc = describe_categorical(df)
        # Create & fit a one hot encoder => encoding to 0, 1 onto several cols
        encoder = OneHotEncoder(drop=None,
                                handle_unknown='error',
                                sparse_output=False,
                                dtype=np.float64,
                                feature_name_combiner=one_hot_name_combiner)
        encoder.fit(df[desc.index.to_list()])
        context['credit_card_multi_cat_features_encoder'] = encoder

    # Encoding multi-class categorical features (n features with value 0, 1)
    # Note that missing values are one-hot encoded in specific features
    encoder = context['credit_card_multi_cat_features_encoder']
    tmpData = encoder.transform(df[encoder.feature_names_in_])
    df.loc[:, encoder.get_feature_names_out()] = tmpData

    # Defragmenting the DataFrame
    df = df.copy()

    return df, context

#### Imputation

In [60]:
def credit_card_imputation(df,
                           relatedDfs,
                           context=None,
                           verbose=True):

    # Local overriding of print
    def _print(*args, **kwargs):
        if 'tabs' in kwargs:
            tabs = int(kwargs['tabs'])
            del kwargs['tabs']
        else:
            tabs = 2
        if verbose:
            print('\t'*tabs, *args, **kwargs)
        else:
            pass

    _print('No imputation for credit_card_balance', tabs=1)

    return df, context

#### Feature engineering

In [61]:
def credit_card_feature_engineering(df,
                                    relatedDfs,
                                    context=None,
                                    verbose=True):

    # Local overriding of print
    def _print(*args, **kwargs):
        if 'tabs' in kwargs:
            tabs = int(kwargs['tabs'])
            del kwargs['tabs']
        else:
            tabs = 2
        if verbose:
            print('\t'*tabs, *args, **kwargs)
        else:
            pass

    _print('No feature engineering for credit_card_balance', tabs=1)

    return df, context

#### Jointure & feature engineering d'agrégation de bureau & bureau_balance

In [62]:
def credit_card_application_merging(mainDf,
                                    secDf,
                                    context=None,
                                    verbose=True):

    # Local overriding of print
    def _print(*args, **kwargs):
        if 'tabs' in kwargs:
            tabs = int(kwargs['tabs'])
            del kwargs['tabs']
        else:
            tabs = 2
        if verbose:
            print('\t'*tabs, *args, **kwargs)
        else:
            pass

    _print('Merging credit_card_balance:',
           'aggregation feature engineering',
           tabs=1)

    # Aggregation w/ feature engineering
    aggRls = {}
    aggRls['MONTHS_BALANCE'] = ['min', 'max', 'mean', 'sum', 'var']
    aggRls['AMT_BALANCE'] = ['min', 'max', 'mean', 'sum', 'var']
    aggRls['AMT_CREDIT_LIMIT_ACTUAL'] = ['min', 'max', 'mean', 'sum', 'var']
    aggRls['AMT_DRAWINGS_ATM_CURRENT'] = ['min', 'max', 'mean', 'sum', 'var']
    aggRls['AMT_DRAWINGS_CURRENT'] = ['min', 'max', 'mean', 'sum', 'var']
    aggRls['AMT_DRAWINGS_OTHER_CURRENT'] = ['min', 'max', 'mean', 'sum', 'var']
    aggRls['AMT_DRAWINGS_POS_CURRENT'] = ['min', 'max', 'mean', 'sum', 'var']
    aggRls['AMT_INST_MIN_REGULARITY'] = ['min', 'max', 'mean', 'sum', 'var']
    aggRls['AMT_PAYMENT_CURRENT'] = ['min', 'max', 'mean', 'sum', 'var']
    aggRls['AMT_PAYMENT_TOTAL_CURRENT'] = ['min', 'max', 'mean', 'sum', 'var']
    aggRls['AMT_RECEIVABLE_PRINCIPAL'] = ['min', 'max', 'mean', 'sum', 'var']
    aggRls['AMT_RECIVABLE'] = ['min', 'max', 'mean', 'sum', 'var']
    aggRls['AMT_TOTAL_RECEIVABLE'] = ['min', 'max', 'mean', 'sum', 'var']
    aggRls['CNT_DRAWINGS_ATM_CURRENT'] = ['min', 'max', 'mean', 'sum', 'var']
    aggRls['CNT_DRAWINGS_CURRENT'] = ['min', 'max', 'mean', 'sum', 'var']
    aggRls['CNT_DRAWINGS_OTHER_CURRENT'] = ['min', 'max', 'mean', 'sum', 'var']
    aggRls['CNT_DRAWINGS_POS_CURRENT'] = ['min', 'max', 'mean', 'sum', 'var']
    aggRls['CNT_INSTALMENT_MATURE_CUM'] = ['min', 'max', 'mean', 'sum', 'var']
    aggRls['SK_DPD'] = ['min', 'max', 'mean', 'sum', 'var']
    aggRls['SK_DPD_DEF'] = ['min', 'max', 'mean', 'sum', 'var']
    for col in [col for col in secDf.columns if col.startswith('ONEHOT__')]:
        aggRls[col] = ['mean']

    secDfGrpby = secDf.groupby('SK_ID_CURR')
    grpdSecDf = secDfGrpby.agg(aggRls)
    # Renaming aggregated columns before merging
    cols = [e[0] + '__CRDCARD_' + e[1].lower() for e in grpdSecDf.columns]
    grpdSecDf.columns = pd.Index(cols)
    grpdSecDf['CRDCARD__CRDCARD_count'] = secDfGrpby.size()

    # Merging
    mainDf = pd.merge(mainDf,
                      grpdSecDf,
                      how='left',
                      left_index=True,
                      right_index=True)

    # Defragmenting the DataFrame
    df = mainDf.copy()

    return df, context

#### Traitement

In [63]:
with timer('Process credit_card_balance'):
    ccbDf, context = credit_card_cleaning_encoding(ccbDf,
                                                   context,
                                                   True)
    ccbDf, context = credit_card_imputation(ccbDf,
                                            {
                                                'application': apDf,
                                                'prev_app': papDf,
                                            },
                                            context,
                                            True)
    ccbDf, context = credit_card_feature_engineering(ccbDf,
                                                     {
                                                         'application': apDf,
                                                         'prev_app': papDf,
                                                     },
                                                     context,
                                                     True)
    apDf, context = credit_card_application_merging(apDf,
                                                    ccbDf,
                                                    context,
                                                    True)
    # Freeing memory
    ccbDf = None
    gc.collect()
print('DataFrame shape:', apDf.shape)

Process credit_card_balance - starting
	 Cleaning & encoding of credit_card_balance
	 No imputation for credit_card_balance
	 No feature engineering for credit_card_balance
	 Merging credit_card_balance: aggregation feature engineering
Process credit_card_balance - done in 19s
DataFrame shape: (356251, 762)


### Sauvegarde du jeu de données et des encoders

In [64]:
papDf = None
_ = gc.collect()

In [65]:
apDf.info()

<class 'pandas.core.frame.DataFrame'>
Index: 356251 entries, 100002 to 456250
Columns: 762 entries, TARGET to CRDCARD__CRDCARD_count
dtypes: float64(708), int64(38), object(16)
memory usage: 2.0+ GB


In [66]:
apDf.to_csv(r'data\dataset.csv', mode='w')

In [67]:
with open(r'data\context.pkl', 'wb') as f:
    pickle.dump(context, f)

## Création du jeu de données avec imputation des valeurs manquantes

* Nous n'allons pas toucher aux features catégorielles car ce sont sur les features encodées correspondantes que les modèles vont travailler.
* Les features catégorielles binaires ont été encodées en features numériques binaires de valeurs 0, 1 ou NaN (features commençant pas 'BINARY__' mais ne se terminant pas par '_mean'). Pour celles-ci, nous allons imputer les valeurs manquantes avec le mode.
* Les features catégorielles ternaires ou plus ont été encodées en n features, la valeur missing faisant l'objet d'une feature dédiée (features commençant pas 'ONEHOT__' mais ne se terminant pas par '_mean'). Pour celles-ci l'imputation n'est pas nécessaire.
* Il restent les features résultant de l'aggrégation de features catégorielles encodées sur plusieurs observations. Pour celles-ci, la seule fonction d'agrégation est la moyenne. Elles commencent pas 'BINARY__' ou 'ONEHOT__' et finissent donc par '_mean'. Dans ce cas, nous allons imputer les valeurs manquantes comme s'il s'agissait de features quantitatives.
* Enfin pour les features quantitatives, nous allons faire l'imputation des valeurs manquantes avec la médiane.
* Dernier point important : même s'ils s'appliquent sur les 2 parties de jeu de données, les modes et médianes sont calculés sur la base du jeu d'entrainement uniquement, afin d'éviter un data leakage.

Vérifions l'exactitude du point 3 :

In [68]:
mask = apDf.columns.str.startswith('ONEHOT__')
mask = mask & ~apDf.columns.str.endswith('_mean')
tmpS = apDf.loc[:, mask].isna().sum()
tmpS[tmpS != 0]

Series([], dtype: int64)

Le taux de valeurs manquantes (hors features catégorielles non encodées)

In [69]:
nanPerc = (apDf.select_dtypes(exclude=['object']).isna().sum() / apDf.shape[0])
nanPerc[nanPerc != 0.0].sort_values(ascending=False)

AMT_DOWN_PAYMENT__PRVAPP_rfusd_min      0.852343
AMT_DOWN_PAYMENT__PRVAPP_rfusd_mean     0.852343
AMT_DOWN_PAYMENT__PRVAPP_rfusd_max      0.852343
RATE_DOWN_PAYMENT__PRVAPP_rfusd_mean    0.852343
RATE_DOWN_PAYMENT__PRVAPP_rfusd_min     0.852343
                                          ...   
AMT_ANNUITY                             0.000101
ANNUITY_INCOME_PERC                     0.000101
CNT_FAM_MEMBERS                         0.000006
INCOME_PER_PERSON                       0.000006
DAYS_LAST_PHONE_CHANGE                  0.000003
Length: 565, dtype: float64

### Imputation des valeurs manquantes pour les features catégorielles encodées

In [70]:
mask = apDf.columns.str.startswith('BINARY__')
mask = mask & ~apDf.columns.str.endswith('_mean')
cols = apDf.columns[mask]
# modes caculated w/o considering test dataset
modes = apDf.loc[~apDf['TARGET'].isna(), cols].mode().iloc[0]
apDf[cols] = apDf[cols].fillna(modes)

### Imputation des valeurs manquantes pour les features quantitatives

In [71]:
# Aggregated categorical features
mask = apDf.columns.str.startswith('BINARY__')
mask = mask | apDf.columns.str.startswith('ONEHOT__')
mask = mask & apDf.columns.str.endswith('_mean')
cols = apDf.columns[mask]
# medians caculated w/o considering test dataset
medians = apDf.loc[~apDf['TARGET'].isna(), cols].median().iloc[0]
apDf[cols] = apDf[cols].fillna(medians)

In [72]:
# Quantitative features
mask = (apDf.columns != 'TARGET')
mask = mask & ~apDf.columns.str.startswith('BINARY__')
mask = mask & ~apDf.columns.str.startswith('ONEHOT__')
mask = mask & apDf.columns.isin(apDf.select_dtypes(exclude=['object']).columns)
cols = apDf.columns[mask]
# medians caculated w/o considering test dataset
medians = apDf.loc[~apDf['TARGET'].isna(), cols].median().iloc[0]
apDf[cols] = apDf[cols].fillna(medians)

Vérifions que nous n'avons plus de valeurs manquantes hormis pour les features catégorielles non encodées et TARGET.

In [73]:
nanPerc = (apDf.select_dtypes(exclude=['object']).isna().sum() / apDf.shape[0])
nanPerc[nanPerc != 0.0].sort_values(ascending=False)

TARGET    0.136825
dtype: float64

### Sauvegarde du jeu de données avec imputation

In [74]:
# Defragmenting the DataFrame
apDf = apDf.copy()
apDf.to_csv(r'data\dataset_no_missing.csv', mode='w')